What this code does:
Sets up a controlled Stable Diffusion (Runway's Stable Diffusion v1.5 model) generation with the bounded attention method.

In [ ]:
!pip install --upgrade pip
!pip install --pre xformers -f https://download.pytorch.org/whl/cu118/torch_stable.html
!pip install pytorch-lightning
!pip install torch torchvision torchaudio --upgrade
!pip install diffusers transformers accelerate safetensors pytorch-lightning
!pip install torch_kmeans
import torch
!pip install torch torchvision torchaudio
!pip install diffusers transformers accelerate safetensors
!pip install numpy matplotlib
import sys
import importlib.util
!pip install nltk
import nltk

Looking in links: https://download.pytorch.org/whl/cu118/torch_stable.html


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
!git clone https://github.com/omer11a/bounded-attention.git
%cd bounded-attention

fatal: destination path 'bounded-attention' already exists and is not an empty directory.
/content/bounded-attention


In [ ]:
!pip install -r requirements.txt

  Using cached accelerate-0.21.0-py3-none-any.whl.metadata (17 kB)
  Using cached diffusers-0.20.0-py3-none-any.whl.metadata (17 kB)
  Using cached einops-0.6.1-py3-none-any.whl.metadata (12 kB)
  Using cached lightning_utilities-0.9.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached matplotlib-3.7.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.7 kB)
  Using cached nltk-3.8.1-py3-none-any.whl.metadata (2.8 kB)
  Using cached opencv_python-4.8.1.78-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached Pillow-9.4.0.tar.gz (50.4 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pytorch_lightning-2.0.7-py3-none-any.whl.metadata (23 kB)
  Using cached scikit_image-0.22.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
  Using cached scikit_learn-1.3.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x

In [ ]:
sys.path.append("/content/bounded-attention")
module_path = "/content/bounded-attention/run_sd.py"
spec = importlib.util.spec_from_file_location("run_sd", module_path)
run_sd = importlib.util.module_from_spec(spec)
sys.modules["run_sd"] = run_sd
spec.loader.exec_module(run_sd)
run = run_sd.run

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from diffusers import StableDiffusionPipeline

In [ ]:
# stable diffusion with BA boxes

model_name = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")
pipe.save_pretrained("sd_model")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Shark and dolphin prompt
from transformers import CLIPTokenizer

prompt = "A shark chasing a dolphin in the ocean"
prompt = "Please show me an image of a shark chasing a dolphin in the ocean"
prompt = "Shark chase dolphin, ocean."
prompt = "A shark with its mouth open, swimming behind and chasing a smiling dolphin in the blue ocean."
prompt = "In the ocean, a shark chasing a dolphin."
prompt = "A shark chasing a dolphin in the ocean, can you show me an image of that?"
prompt = "A dolphin being chased by a shark in the ocean"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

subject_token_indices = [[2,3,4,5,6], [13,14]] #shark, dolphin

boxes = [
    [0.1, 0.2, 0.45, 0.8],  # shark
    [0.55, 0.2, 0.9, 0.8]   # dolphin
]

In [ ]:
# flowers prompt
from transformers import CLIPTokenizer

prompt = "A porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers"
prompt = "Please show me an image of a porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers"
prompt = "Porcelain pot tulips, metal can orchids, glass jar sunflowers."
prompt = "A white porcelain pot with pink tulips and a silver metal can with purple orchids and a clear glass jar with yellow sunflowers, next to each other on a wooden table"
prompt = "In the kitchen, a porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers"
prompt = "A porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers, can you show me an image of that?"
prompt = "Tulips in a porcelain pot and orchids in a metal can and sunflowers in a glass jar"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

subject_token_indices = [[4,5], [1], [10,11], [7], [16,17], [13]]

boxes = [
    [0.1, 0.5, 0.25, 0.85],     # pot1
    [0.05, 0.05, 0.3, 0.5],     # flower1
    [0.45, 0.45, 0.6, 0.8],     # pot2
    [0.37, 0.15, 0.65, 0.45],   # flower2
    [0.75, 0.65, 0.9, 0.95],    # pot3
    [0.7, 0.1, 0.95, 0.65] ]     # flower3

In [ ]:
# cartoon people prompt
from transformers import CLIPTokenizer

prompt = "3D Pixar animation of a princess, an elf, and a fairy in a castle"
prompt = "Please show me a 3D Pixar animation of a princess, an elf, and a fairy in a castle"
prompt = "3D Pixar animation, princess, elf, fairy, castle"
prompt = "A 3D Pixar animation of a blonde princess wearing a pink dress next to a fat elf with pointed ears with a tan fairy wearing a green dress in a diamond castle"
prompt = "In a castle, a princess, an elf, and a fairy 3D Pixar animation"
prompt = "A princess, an elf, and a fairy in a castle, can you show me a 3D Pixar animation of that?"
prompt = "3D Pixar animation of a fairy, an elf, and a princess in a castle"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

subject_token_indices = [[7],[10],[14]]

boxes = [
    [0.1, 0.1, 0.3, 0.85],     # princess
    [0.4, 0.45, 0.7, 0.85],     # elf
    [0.7, 0.1, 0.85, 0.35]      # fairy
]

In [ ]:
# realistic person, solo prompt
from transformers import CLIPTokenizer

prompt = "A hyperrealistic jedi with a lightsaber in the death star."
prompt = "Please show me a hyperrealistic jedi with a lightsaber in the death star."
prompt = "Hyperrealistic jedi, lightsaber, death star."
prompt = "A hyperrealistic jedi with a yellow lightsaber and long hair at emperor palpatine’s throne in the death star."
prompt = "In the death star, a hyperrealistic jedi with a lightsaber"
prompt = "A hyperrealistic jedi with a lightsaber, can you show me an image of that?"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

subject_token_indices = [[2,3,4,5,6,7,8]]

boxes = [[0.4, 0.3, 0.6, 0.85]]

0 <|startoftext|>
1 a
2 hyper
3 realistic
4 jedi
5 with
6 a
7 light
8 saber
9 ,
10 can
11 you
12 show
13 me
14 an
15 image
16 of
17 that
18 ?
19 <|endoftext|>


In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
# Bounded Attention with SD
import sys
import importlib.util

sys.path.append("/content/bounded-attention")

module_path = "/content/bounded-attention/run_sd.py"
spec = importlib.util.spec_from_file_location("run_sd", module_path)
run_sd = importlib.util.module_from_spec(spec)
sys.modules["run_sd"] = run_sd
spec.loader.exec_module(run_sd)

run = run_sd.run

run(
    boxes=boxes,
    prompt=prompt,
    subject_token_indices=subject_token_indices,
    out_dir="outputs",
    seed=42,
    batch_size=1,
    init_step_size=25,
    final_step_size=10
)

Keyword arguments {'cross_attention_kwargs': {'scale': 0.5}} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

...

Full batch converged at iteration 2/100 with center shifts = tensor([0.], device='cuda:0', dtype=torch.float16).


DDIM Sampler:  90%|█████████ | 45/50 [02:29<00:05,  1.05s/it]

Full batch converged at iteration 5/100 with center shifts = tensor([0.], device='cuda:0', dtype=torch.float16).


DDIM Sampler: 100%|██████████| 50/50 [02:34<00:00,  3.09s/it]


Syntheiszed images are saved in outputs/sample_2
